### NLP GRP 12

### 1. Imports

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import warnings
import re
import html
from bs4 import BeautifulSoup
import json


warnings.filterwarnings('ignore')

### 2. Loading Dataset

In [2]:
import json

RAW_DIR = Path('data/raw')
print("Working dir:", Path.cwd())

# --- Re-DocRED: each file is a single JSON array ---
redocred_files = {'train': 'train_revised.json', 'validation': 'dev_revised.json', 'test': 'test_revised.json'}
redocred = {split: pd.read_json(RAW_DIR / 'REDocRed' / f) for split, f in redocred_files.items()}

# --- SciERC: JSON Lines (one document per line) ---
scierc_files = {'train': 'train.json', 'validation': 'dev.json', 'test': 'test.json'}
scierc = {split: pd.read_json(RAW_DIR / 'sciREC' / f, lines=True) for split, f in scierc_files.items()}

# --- DWIE: one JSON file per article; split is stored in the 'tags' field ---
# Prefer annos_with_content (has article text, created by dwie_download.py); fall back to annos-only
DWIE_DATA = RAW_DIR / 'DWIE' / 'data'
dwie_dir = DWIE_DATA / 'annos_with_content'
n_annos = len(list((DWIE_DATA / 'annos').glob('*.json')))
if len(list(dwie_dir.glob('*.json'))) < n_annos:
    print("annos_with_content incomplete -> loading annos only (no 'content' column)")
    dwie_dir = DWIE_DATA / 'annos'

dwie_files = sorted(dwie_dir.glob('*.json'))
if not dwie_files:
    raise FileNotFoundError(f"No DWIE json files found in {dwie_dir.resolve()}")

dwie_records = []
for fp in dwie_files:
    with open(fp, encoding='utf-8') as f:
        dwie_records.append(json.load(f))
dwie_all = pd.DataFrame(dwie_records)
dwie = {split: dwie_all[dwie_all['tags'].apply(lambda t: split in t)].reset_index(drop=True)
        for split in ['train', 'test']}

datasets = {'Re-DocRED': redocred, 'SciERC': scierc, 'DWIE': dwie}

for ds_name, splits in datasets.items():
    for split, d in splits.items():
        print(f"{ds_name:10s} {split:10s} {d.shape}")

Working dir: c:\Users\Iris\NLP_grp12
Re-DocRED  train      (3053, 4)
Re-DocRED  validation (500, 4)
Re-DocRED  test       (500, 4)
SciERC     train      (350, 5)
SciERC     validation (50, 5)
SciERC     test       (100, 5)
DWIE       train      (702, 8)
DWIE       test       (100, 8)


In [15]:
from IPython.display import display

pd.set_option('display.max_colwidth', 200)

for ds_name, splits in datasets.items():
    for split, d in splits.items():
        print(f"\n===== {ds_name} | {split.upper()} {d.shape} =====")
        print(d.columns.tolist())
        display(d.head())


===== Re-DocRED | TRAIN (3053, 4) =====
['title', 'vertexSet', 'labels', 'sents']


,title,vertexSet,labels,sents
0,AirAsia Zest,"[[{'pos': [0, 4], 'type': 'ORG', 'sent_id': 0, 'name': 'Zest Airways, Inc.', 'global_pos': [0, 0], 'index': '0_0'}, {'sent_id': 0, 'type': 'ORG', 'pos': [10, 15], 'name': 'Asian Spirit and Zest Ai...","[{'r': 'P159', 'h': 0, 't': 2, 'evidence': [0]}, {'r': 'P17', 'h': 0, 't': 4, 'evidence': [2, 4, 7]}, {'r': 'P17', 'h': 12, 't': 4, 'evidence': [6, 7]}, {'r': 'P17', 'h': 2, 't': 4, 'evidence': [0...","[[Zest, Airways, ,, Inc., operated, as, AirAsia, Zest, (, formerly, Asian, Spirit, and, Zest, Air, ), ,, was, a, low, -, cost, airline, based, at, the, Ninoy, Aquino, International, Airport, in, P..."
1,Short-beaked common dolphin,"[[{'pos': [7, 9], 'type': 'MISC', 'name': 'Delphinus delphis', 'sent_id': 0, 'global_pos': [7, 7], 'index': '0_0'}, {'pos': [1, 6], 'type': 'MISC', 'name': 'short - beaked common dolphin', 'sent_i...","[{'h': 5, 't': 3, 'r': 'P361', 'evidence': []}, {'h': 3, 't': 5, 'r': 'P527', 'evidence': []}]","[[The, short, -, beaked, common, dolphin, (, Delphinus, delphis, ), is, a, species, of, common, dolphin, .], [It, has, a, larger, range, than, the, long, -, beaked, common, dolphin, (, D., capensi..."
2,Niklas Bergqvist,"[[{'name': 'Bergqvist', 'pos': [9, 10], 'sent_id': 5, 'type': 'PER', 'global_pos': [116, 116], 'index': '0_0'}, {'name': 'Niklas Bergqvist', 'pos': [0, 2], 'sent_id': 0, 'type': 'PER', 'global_pos...","[{'r': 'P17', 'h': 2, 't': 3, 'evidence': [0]}, {'r': 'P27', 'h': 11, 't': 3, 'evidence': [4]}, {'r': 'P569', 'h': 0, 't': 1, 'evidence': [0]}, {'r': 'P19', 'h': 0, 't': 2, 'evidence': [0]}, {'r':...","[[Niklas, Bergqvist, (, born, 6, October, 1962, in, Stockholm, ), ,, is, a, Swedish, songwriter, ,, producer, and, musician, .], [His, career, started, in, 1984, ,, as, a, founding, member, of, th..."
3,Pacific Fair Shopping Centre,"[[{'name': 'Pacific Fair', 'pos': [0, 2], 'sent_id': 8, 'type': 'LOC', 'global_pos': [168, 168], 'index': '0_0'}, {'sent_id': 2, 'type': 'LOC', 'pos': [0, 2], 'name': 'Pacific Fair', 'global_pos':...","[{'r': 'P131', 'h': 3, 't': 4, 'evidence': [0]}, {'r': 'P17', 'h': 3, 't': 4, 'evidence': [0, 5, 6]}, {'r': 'P150', 'h': 3, 't': 2, 'evidence': [0, 1, 4]}, {'r': 'P150', 'h': 4, 't': 3, 'evidence'...","[[Pacific, Fair, is, a, major, shopping, centre, in, Broadbeach, Waters, on, the, Gold, Coast, ,, Queensland, ,, Australia, .], [It, was, Queensland, 's, largest, regional, shopping, centre, until..."
4,Ministry of Home Security,"[[{'pos': [1, 5], 'type': 'ORG', 'sent_id': 1, 'name': 'Ministry for Home Security', 'global_pos': [34, 34], 'index': '0_0'}, {'name': 'Ministry of Home Security', 'pos': [1, 5], 'sent_id': 0, 'ty...","[{'r': 'P571', 'h': 0, 't': 2, 'evidence': [0]}, {'r': 'P576', 'h': 0, 't': 14, 'evidence': [7]}, {'r': 'P607', 'h': 4, 't': 3, 'evidence': [0, 1]}, {'r': 'P27', 'h': 4, 't': 1, 'evidence': [0, 1]...","[[The, Ministry, of, Home, Security, was, a, British, government, department, established, in, 1939, to, direct, national, civil, defence, (, primarily, tasked, with, organising, air, raid, precau..."



===== Re-DocRED | VALIDATION (500, 4) =====
['title', 'vertexSet', 'labels', 'sents']


,title,vertexSet,labels,sents
0,Willi Schneider (skeleton racer),"[[{'type': 'PER', 'pos': [0, 5], 'name': 'Wilfried "" Willi "" Schneider', 'sent_id': 0, 'global_pos': [0, 0], 'index': '0_0'}, {'name': 'Schneider', 'pos': [4, 5], 'sent_id': 4, 'type': 'PER', 'glo...","[{'r': 'P580', 'h': 11, 't': 6, 'evidence': [2]}, {'r': 'P582', 'h': 11, 't': 6, 'evidence': [2]}, {'r': 'P276', 'h': 11, 't': 12, 'evidence': [2]}, {'r': 'P276', 'h': 16, 't': 17, 'evidence': [4]...","[[Wilfried, "", Willi, "", Schneider, (, born, 13, March, 1963, in, Mediaș, ,, Transylvania, ), is, a, German, skeleton, racer, who, competed, from, 1992, to, 2002, .], [He, won, two, medals, in, th..."
1,Ross Alger,"[[{'name': 'Ross Patterson Alger', 'pos': [0, 3], 'sent_id': 0, 'type': 'PER', 'global_pos': [0, 0], 'index': '0_0'}, {'name': 'Alger', 'pos': [1, 2], 'sent_id': 10, 'type': 'PER', 'global_pos': [...","[{'r': 'P19', 'h': 0, 't': 8, 'evidence': [0, 1]}, {'r': 'P69', 'h': 0, 't': 11, 'evidence': [0, 2]}, {'r': 'P69', 'h': 0, 't': 15, 'evidence': [0, 4]}, {'r': 'P3373', 'h': 0, 't': 24, 'evidence':...","[[Ross, Patterson, Alger, (, August, 20, ,, 1920, –, January, 16, ,, 1992, ), was, a, politician, in, the, Canadian, province, of, Alberta, ,, who, served, as, mayor, of, Calgary, from, 1977, to, ..."
2,Mess of Blues (Jeff Healey album),"[[{'name': 'Mess of Blues', 'pos': [3, 6], 'sent_id': 5, 'type': 'MISC', 'global_pos': [112, 112], 'index': '0_0'}, {'type': 'MISC', 'pos': [0, 3], 'name': 'Mess of Blues', 'sent_id': 0, 'global_p...","[{'r': 'P577', 'h': 0, 't': 2, 'evidence': [0, 1]}, {'r': 'P175', 'h': 0, 't': 1, 'evidence': [0, 4]}, {'r': 'P175', 'h': 0, 't': 19, 'evidence': [5]}, {'h': 10, 't': 13, 'r': 'P17', 'evidence': [...","[[Mess, of, Blues, is, an, album, by, Jeff, Healey, .], [It, was, released, in, 2008, less, than, two, weeks, after, his, death, and, just, three, weeks, shy, of, his, 42nd, birthday, .], [Four, o..."
3,Ramey Idriss,"[[{'name': 'Ramey Idriss', 'pos': [0, 2], 'sent_id': 0, 'type': 'PER', 'global_pos': [0, 0], 'index': '0_0'}, {'pos': [4, 6], 'type': 'PER', 'sent_id': 1, 'name': 'Ramez Idriss', 'global_pos': [33...","[{'r': 'P577', 'h': 13, 't': 14, 'evidence': [3]}, {'r': 'P69', 'h': 0, 't': 4, 'evidence': [0]}, {'r': 'P27', 'h': 0, 't': 3, 'evidence': [0]}, {'r': 'P569', 'h': 0, 't': 1, 'evidence': [0]}, {'r...","[[Ramey, Idriss, (, 11, September, 1911, –, 5, February, 1971, ), was, an, American, songwriter, ,, author, ,, composer, and, musician, ,, educated, at, Los, Angeles, Community, College, .], [His,..."
4,ELAM (Latin American School of Medicine) Cuba,"[[{'pos': [0, 4], 'type': 'ORG', 'sent_id': 0, 'name': 'Escuela Latinoamericana de Medicina', 'global_pos': [0, 0], 'index': '0_0'}], [{'pos': [9, 14], 'type': 'ORG', 'sent_id': 0, 'name': 'Escuel...","[{'r': 'P17', 'h': 4, 't': 5, 'evidence': [0, 2]}, {'r': 'P571', 'h': 4, 't': 7, 'evidence': [0, 1]}, {'r': 'P571', 'h': 0, 't': 7, 'evidence': [0, 1]}, {'r': 'P571', 'h': 1, 't': 7, 'evidence': [...","[[Escuela, Latinoamericana, de, Medicina, (, ELAM, ), ,, formerly, Escuela, Latinoamericana, de, Ciencias, Médicas, (, in, Spanish, ;, in, English, :, Latin, American, School, of, Medicine, (, LAS..."



===== Re-DocRED | TEST (500, 4) =====
['title', 'vertexSet', 'labels', 'sents']


,title,vertexSet,labels,sents
0,Loud Tour,"[[{'name': 'Loud', 'pos': [23, 24], 'sent_id': 1, 'type': 'MISC', 'global_pos': [41, 41], 'index': '0_0'}, {'name': 'Loud Tour', 'pos': [1, 3], 'sent_id': 6, 'type': 'MISC', 'global_pos': [128, 12...","[{'r': 'P577', 'h': 0, 't': 6, 'evidence': [1]}, {'r': 'P175', 'h': 0, 't': 2, 'evidence': [0, 1]}, {'r': 'P131', 'h': 10, 't': 8, 'evidence': [4]}, {'r': 'P17', 'h': 8, 't': 7, 'evidence': [3, 4]...","[[The, Loud, Tour, was, the, fourth, overall, and, third, world, concert, tour, by, Barbadian, recording, artist, Rihanna, .], [Performing, in, over, twenty, countries, in, the, Americas, and, Eur..."
1,Vladimir Mitrofanovich Orlov,"[[{'name': 'Vladimir Mitrofanovich Orlov', 'pos': [0, 3], 'sent_id': 0, 'type': 'PER', 'global_pos': [0, 0], 'index': '0_0'}, {'name': 'Orlov', 'pos': [0, 1], 'sent_id': 7, 'type': 'PER', 'global_...","[{'r': 'P69', 'h': 0, 't': 9, 'evidence': [1]}, {'r': 'P570', 'h': 0, 't': 25, 'evidence': [0, 7]}, {'r': 'P19', 'h': 0, 't': 7, 'evidence': [1]}, {'r': 'P569', 'h': 0, 't': 1, 'evidence': [0]}, {...","[[Vladimir, Mitrofanovich, Orlov, (, ), (, July, 15, ,, 1895, -, July, 28, ,, 1938, ), was, a, Russian, military, leader, and, Commander, -, in, -, Chief, of, the, Soviet, Naval, Forces, from, Jul..."
2,Ecuadorian Constituent Assembly,"[[{'name': 'Ecuadorian Constituent Assembly', 'pos': [1, 4], 'sent_id': 0, 'type': 'ORG', 'global_pos': [1, 1], 'index': '0_0'}], [{'type': 'TIME', 'pos': [6, 7], 'name': '2007', 'sent_id': 0, 'gl...","[{'r': 'P17', 'h': 0, 't': 3, 'evidence': [0]}, {'r': 'P571', 'h': 0, 't': 1, 'evidence': [0]}, {'r': 'P488', 'h': 12, 't': 11, 'evidence': [3]}, {'r': 'P17', 'h': 12, 't': 3, 'evidence': [0, 1, 3...","[[The, Ecuadorian, Constituent, Assembly, was, a, 2007, –, 2008, constitutional, assembly, in, Ecuador, ,, which, drafted, the, 2008, Constitution, of, Ecuador, ,, approved, via, the, Ecuadorian, ..."
3,Bantustan,"[[{'name': 'Bantustan', 'pos': [1, 2], 'sent_id': 0, 'type': 'LOC', 'global_pos': [1, 1], 'index': '0_0'}], [{'sent_id': 0, 'type': 'LOC', 'pos': [6, 8], 'name': 'Bantu homeland', 'global_pos': [6...","[{'r': 'P1366', 'h': 3, 't': 4, 'evidence': [0]}, {'r': 'P1365', 'h': 4, 't': 3, 'evidence': [0]}, {'r': 'P576', 'h': 16, 't': 26, 'evidence': [8, 5]}, {'r': 'P17', 'h': 17, 't': 2, 'evidence': [8...","[[A, Bantustan, (, also, known, as, Bantu, homeland, ,, black, homeland, ,, black, state, or, simply, homeland, ;, ), was, a, territory, set, aside, for, black, inhabitants, of, South, Africa, and..."
4,Long Hard Road Out of Hell,"[[{'name': 'Long Hard Road Out of Hell', 'pos': [1, 7], 'sent_id': 0, 'type': 'MISC', 'global_pos': [1, 1], 'index': '0_0'}, {'name': 'Long Hard Road Out of Hell', 'pos': [9, 15], 'sent_id': 1, 't...","[{'r': 'P162', 'h': 0, 't': 7, 'evidence': [0, 1, 5]}, {'r': 'P162', 'h': 0, 't': 2, 'evidence': [0, 1, 5]}, {'r': 'P175', 'h': 0, 't': 2, 'evidence': [0, 1, 5]}, {'r': 'P175', 'h': 0, 't': 4, 'ev...","[["", Long, Hard, Road, Out, of, Hell, "", is, a, song, by, American, rock, band, Marilyn, Manson, and, British, trip, hop, band, Sneaker, Pimps, ,, released, as, a, single, from, the, soundtrack, t..."


... (6 outputs left out)


===== DWIE | TRAIN (702, 8) =====
['id', 'content', 'tags', 'mentions', 'concepts', 'relations', 'frames', 'iptc']


,id,content,tags,mentions,concepts,relations,frames,iptc
0,DW_1095032,"Mzoudi Verdict Postponed Amid New Testimony\nAl Qaeda is allegedly trying to assassinate Abdelghani Mzoudi, who is on trial in Germany for his suspected involvement in the Sept. 11 attacks, a witn...","[all, train]","[{'begin': 0, 'end': 6, 'text': 'Mzoudi', 'concept': 2, 'candidates': [], 'scores': []}, {'begin': 44, 'end': 52, 'text': 'Al Qaeda', 'concept': 1, 'candidates': ['Al-Qaeda', 'Tanzim_Qaidat_al-Jih...","[{'concept': 0, 'text': 'Germany', 'keyword': True, 'count': 3, 'link': 'Germany', 'tags': ['slot::in', 'slot::keyword', 'topic::none', 'type::entity', 'type::gpe', 'type::gpe0', 'type::location']...","[{'s': 2, 'p': 'citizen_of', 'o': 13}, {'s': 6, 'p': 'in0', 'o': 0}, {'s': 6, 'p': 'in0-x', 'o': 12}, {'s': 7, 'p': 'gpe0', 'o': 14}, {'s': 12, 'p': 'gpe0', 'o': 0}]","[{'type': 'trial-hearing', 'slots': [{'name': 'defendant', 'value': 2}, {'name': 'place', 'value': 0}, {'name': 'time', 'value': 4}]}]","[02000000, 16000000, 20000053, 20000106, 20000107, 20000111]"
1,DW_1213203,"Eastern Germany's EU Funds in Danger of Drying Up\nThe EU's competition commissioner met with the heads of Germany's eastern states to discuss future financial support for the region, which may no...","[all, train]","[{'begin': 8, 'end': 15, 'text': 'Germany', 'concept': 4, 'candidates': ['Germany', 'Germany_national_football_team', 'Nazi_Germany', 'German_Empire', 'Germany_women's_national_football_team', 'We...","[{'concept': 0, 'text': 'East Germany', 'keyword': True, 'count': 2, 'link': 'East_Germany', 'tags': ['slot::keyword', 'topic::none', 'type::entity', 'type::gpe', 'type::gpe0', 'type::location']},...","[{'s': 2, 'p': 'member_of', 'o': 1}, {'s': 3, 'p': 'agent_of', 'o': 5}, {'s': 3, 'p': 'citizen_of', 'o': 4}, {'s': 3, 'p': 'citizen_of-x', 'o': 10}, {'s': 3, 'p': 'head_of_gov', 'o': 5}, {'s': 5, ...","[{'type': 'todo', 'slots': []}]","[04000000, 20000344, 20000346, 20000367]"
2,DW_1269079,"German Know-How For China's Energy Sector\nEnergy is becoming a scarce resource in China and Beijing is looking for alternatives. On Friday, Germany's foreign minister inaugurated one of the large...","[all, train]","[{'begin': 0, 'end': 6, 'text': 'German', 'concept': 4}, {'begin': 20, 'end': 25, 'text': 'China', 'concept': 0, 'candidates': ['China', 'China_national_football_team', 'Republic_of_China_(1912–19...","[{'concept': 0, 'text': 'China', 'keyword': True, 'count': 9, 'link': 'China', 'tags': ['slot::in', 'slot::keyword', 'topic::none', 'type::entity', 'type::gpe', 'type::gpe0', 'type::location']}, {...","[{'s': 1, 'p': 'based_in0', 'o': 7}, {'s': 1, 'p': 'based_in0-x', 'o': 4}, {'s': 2, 'p': 'head_of', 'o': 1}, {'s': 2, 'p': 'head_of', 'o': 17}, {'s': 2, 'p': 'member_of', 'o': 1}, {'s': 2, 'p': 'm...","[{'type': 'todo', 'slots': []}, {'type': 'cpn-business', 'slots': []}]","[04000000, 20000209, 20000256, 20000257]"
3,DW_1272404,"German Reality Shows in Need of Reality Check\nReality shows are increasingly attracting the ire of critics -- some who object to the way animals are treated on a new farm docu-soap, and others wh...","[all, train]","[{'begin': 0, 'end': 6, 'text': 'German', 'concept': 5}, {'begin': 265, 'end': 272, 'text': 'Die Alm', 'concept': 2, 'candidates': [], 'scores': []}, {'begin': 278, 'end': 286, 'text': 'The Alps',...","[{'concept': 0, 'text': 'German Society for the Protection of Animals', 'keyword': True, 'count': 3, 'link': 'wikidata:Q1205394', 'tags': ['iptc::11000000', 'slot::keyword', 'topic::politics', 'ty...","[{'s': 0, 'p': 'based_in0', 'o': 3}, {'s': 0, 'p': 'based_in0-x', 'o': 5}, {'s': 1, 'p': 'based_in0', 'o': 3}, {'s': 1, 'p': 'based_in0-x', 'o': 5}, {'s': 1, 'p': 'based_in2', 'o': 6}, {'s': 4, 'p...","[{'type': 'none', 'slots': [{'name': 'place', 'value': 3}]}]","[14000000, 20000808, 20000814]"
4,DW_1293372,"Baltic Beach Battle\nThe Baltic Sea stretching from eastern Germany to Poland used to be a Mecca for beach bums who prefer skin to


===== DWIE | TEST (100, 8) =====
['id', 'content', 'tags', 'mentions', 'concepts', 'relations', 'frames', 'iptc']


,id,content,tags,mentions,concepts,relations,frames,iptc
0,DW_14739175,"For drivers in Germany, 2010 was the most expensive year ever\nOwning and operating a car in Germany has never been cheap, but it's never been more expensive than in 2010. German auto club ADAC sa...","[all, test]","[{'begin': 15, 'end': 22, 'text': 'Germany', 'concept': 0, 'candidates': ['Germany', 'Germany_national_football_team', 'Nazi_Germany', 'German_Empire', 'Germany_women's_national_football_team', 'W...","[{'concept': 0, 'text': 'Germany', 'keyword': True, 'count': 2, 'link': 'Germany', 'tags': ['slot::in', 'slot::keyword', 'topic::none', 'type::entity', 'type::gpe', 'type::gpe0', 'type::location']...","[{'s': 1, 'p': 'based_in0', 'o': 0}, {'s': 1, 'p': 'based_in0-x', 'o': 3}, {'s': 3, 'p': 'gpe0', 'o': 0}]","[{'type': 'none', 'slots': []}]","[04000000, 20000209, 20000337, 20000341]"
1,DW_14777482,"Belarusian president accuses Germany, Poland of coup plans\nBefore being sworn in for a fourth term, the reclusive leader of Belarus fired a warning salvo at the European Union, accusing Germany a...","[all, test]","[{'begin': 0, 'end': 10, 'text': 'Belarusian', 'concept': 7}, {'begin': 29, 'end': 36, 'text': 'Germany', 'concept': 3, 'candidates': ['Germany', 'Germany_national_football_team', 'Nazi_Germany', ...","[{'concept': 0, 'text': 'Western', 'keyword': False, 'count': 2, 'link': 'Western_world', 'tags': ['topic::none', 'type::entity', 'type::loc-x', 'type::location', 'type::other', 'type::regio']}, {...","[{'s': 2, 'p': 'agent_of', 'o': 5}, {'s': 2, 'p': 'citizen_of', 'o': 5}, {'s': 2, 'p': 'citizen_of-x', 'o': 7}, {'s': 2, 'p': 'head_of_state', 'o': 5}, {'s': 2, 'p': 'head_of_state-x', 'o': 7}, {'...","[{'type': 'start-position', 'slots': [{'name': 'person', 'value': 2}, {'name': 'time', 'value': 9}, {'name': 'place', 'value': 5}]}, {'type': 'elect', 'slots': []}, {'type': 'todo', 'slots': []}]","[11000000, 20000638]"
2,DW_14858431,White House ignored multiple warnings about crisis in Egypt\nThere's a general consensus among policymakers and analysts that the uprisings in Egypt and elsewhere were unpredictable. That's untrue...,"[all, test]","[{'begin': 0, 'end': 11, 'text': 'White House', 'concept': 3, 'candidates': ['White_House', 'Executive_Office_of_the_President_of_the_United_States', 'White_House_Office', 'White_House_(Moscow)', ...","[{'concept': 0, 'text': 'United States', 'keyword': False, 'count': 8, 'link': 'United_States', 'tags': ['topic::none', 'type::entity', 'type::gpe', 'type::gpe0', 'type::location']}, {'concept': 1...","[{'s': 1, 'p': 'in0', 'o': 0}, {'s': 1, 'p': 'in0', 'o': 0}, {'s': 3, 'p': 'agency_of', 'o': 0}, {'s': 3, 'p': 'based_in0', 'o': 0}, {'s': 3, 'p': 'in0', 'o': 0}, {'s': 4, 'p': 'agent_of', 'o': 2}...","[{'type': 'todo', 'slots': []}]","[11000000, 20000647]"
3,DW_14929074,"Berlin court rules Google Street View is legal in Germany\nA Berlin court has ruled that Google Street View is legal. But that doesn't mean the lawsuits will end, legal scholars say.\nA Berlin cou...","[all, test]","[{'begin': 0, 'end': 6, 'text': 'Berlin', 'concept': 3, 'candidates': ['Berlin', 'Humboldt_University_of_Berlin', 'Berlin_(band)', 'German_Open_(WTA)', 'Berlin,_New_Hampshire', 'Berlin_Hauptbahnho...","[{'concept': 0, 'text': 'Google', 'keyword': True, 'count': 8, 'link': 'Google', 'tags': ['iptc::20000170', 'sector::20000225', 'slot::keyword', 'topic::business', 'type::company', 'type::entity',...","[{'s': 1, 'p': 'product_of', 'o': 0}, {'s': 2, 'p': 'based_in0', 'o': 4}, {'s': 2, 'p': 'based_in0-x', 'o': 6}, {'s': 2, 'p': 'based_in0-x', 'o': 7}, {'s': 2, 'p': 'based_in2', 'o': 3}, {'s': 3, '...","[{'type': 'acquit', 'slots': [{'name': 'defendant', 'value': 0}, {'name': 'place', 'value': 3}]}]","[02000000, 04000000, 11000000, 20000106, 20000107, 20000119, 20000120, 20000170, 20000209, 20000225, 20000587]"
4,DW_14959026,"In Argentina, red wine exports to balance Porsche imports\nAbout 100 Porsches are imported to Argentina each yea